In [8]:
import pandas as pd

# đọc file
df = pd.read_csv("clean_hospital_data.csv")

# convert datetime
df['ActualArrivalTime'] = pd.to_datetime(df['ActualArrivalTime'])
df['DischargeTime'] = pd.to_datetime(df['DischargeTime'])

# tạo hour
df['Hour'] = df['ActualArrivalTime'].dt.hour

# ==============================
# 🔵 1. PATIENTS IN SYSTEM (theo giờ)
# ==============================

hours = list(range(24))
results = []

for date in df['ActualArrivalTime'].dt.date.unique():
    df_day = df[df['ActualArrivalTime'].dt.date == date]

    for h in hours:
        hour_start = pd.Timestamp(date) + pd.Timedelta(hours=h)
        hour_end = hour_start + pd.Timedelta(hours=1)

        count = df_day[
            (df_day['ActualArrivalTime'] < hour_end) &
            (df_day['DischargeTime'] >= hour_start)
        ]['PatientID'].nunique()

        results.append({
            'VisitDate': date,
            'Hour': h,
            'Patients': count
        })

patients_df = pd.DataFrame(results)

# ==============================
# 🔵 2. NURSES theo giờ + ngày
# ==============================

nurse_daily = df.groupby(['ActualArrivalTime', 'Hour'])['NursesOnShift'] \
    .mean().reset_index()

# ⚠️ sửa đúng key: phải theo ngày
df['VisitDate'] = df['ActualArrivalTime'].dt.date

nurse_daily = df.groupby(['VisitDate', 'Hour'])['NursesOnShift'] \
    .mean().reset_index(name='Nurses')

# ==============================
# 🔵 3. MERGE
# ==============================

daily = pd.merge(patients_df, nurse_daily, on=['VisitDate', 'Hour'])

# ==============================
# 🔵 4. TÍNH RATIO TRƯỚC
# ==============================

daily['Patients_per_Nurse'] = daily['Patients'] / daily['Nurses']

# ==============================
# 🔵 5. AGG THEO KHUNG GIỜ
# ==============================

final = daily.groupby('Hour').agg({
    'Patients': 'mean',
    'Nurses': 'mean',
    'Patients_per_Nurse': 'median'
}).reset_index()

# ==============================
# 🔵 6. EXPORT
# ==============================

final.to_csv("patients_vs_nurse_corrected_hour.csv", index=False)

print(final)

    Hour   Patients    Nurses  Patients_per_Nurse
0      7   1.951220  8.990447            0.200000
1      8   8.076923  9.266574            0.851351
2      9  13.758242  8.829586            1.577465
3     10  18.340659  9.110376            2.022222
4     11  21.098901  8.905750            2.402985
5     12  22.406593  9.088947            2.555556
6     13  22.692308  8.771188            2.608696
7     14  22.900000  9.036478            2.460277
8     15  22.747253  9.172359            2.500000
9     16  20.988889  9.104775            2.292819
10    17  15.400000  9.900000            1.714286


In [2]:
df['ProviderStartTime'] = pd.to_datetime(df['ProviderStartTime'])
df['ProviderEndTime'] = pd.to_datetime(df['ProviderEndTime'])

hours = list(range(24))
results = []

for h in hours:
    for dept in df['Department'].unique():
        counts = []

        for date in df['ProviderStartTime'].dt.date.unique():
            hour_start = pd.Timestamp(date) + pd.Timedelta(hours=h)
            hour_end = hour_start + pd.Timedelta(hours=1)

            count = df[
                (df['Department'] == dept) &
                (df['ProviderStartTime'] < hour_end) &
                (df['ProviderEndTime'] >= hour_start)
            ]['ProviderID'].nunique()

            counts.append(count)

        results.append({
            'Department': dept,
            'Hour': h,
            'Avg_Providers': sum(counts) / len(counts)
        })

dept_providers = pd.DataFrame(results)

In [3]:
df['ProviderStartTime'] = pd.to_datetime(df['ProviderStartTime'])
df['ProviderEndTime'] = pd.to_datetime(df['ProviderEndTime'])

hours = list(range(24))
results = []

for h in hours:
    for dept in df['Department'].unique():
        counts = []

        for date in df['ProviderStartTime'].dt.date.unique():
            hour_start = pd.Timestamp(date) + pd.Timedelta(hours=h)
            hour_end = hour_start + pd.Timedelta(hours=1)

            count = df[
                (df['Department'] == dept) &
                (df['ProviderStartTime'] < hour_end) &
                (df['ProviderEndTime'] >= hour_start)
            ]['ProviderID'].nunique()

            counts.append(count)

        results.append({
            'Department': dept,
            'Hour': h,
            'Avg_Providers': sum(counts) / len(counts)
        })

dept_providers = pd.DataFrame(results)
dept_providers

,Department,Hour,Avg_Providers
0,Orthopedics,0,0.0
1,Cardiology,0,0.0
2,General Surgery,0,0.0
3,Emergency,0,0.0
4,Radiology,0,0.0
...,...,...,...
235,Obstetrics,23,0.0
236,Neurology,23,0.0
237,Oncology,23,0.0
238,Pediatrics,23,0.0


In [4]:
# tạo date + hour
df['VisitDate'] = df['ActualArrivalTime'].dt.date
df['Hour'] = df['ActualArrivalTime'].dt.hour

# ==============================
# 🔵 1. PATIENTS theo ngày + giờ + khoa
# ==============================

patients = df.groupby(['VisitDate', 'Department', 'Hour'])['PatientID'] \
    .nunique().reset_index()

patients.rename(columns={'PatientID': 'Patients'}, inplace=True)

# ==============================
# 🔵 2. PROVIDERS theo ngày + giờ + khoa
# ==============================

providers = df.groupby(['VisitDate', 'Department', 'Hour'])['ProviderID'] \
    .nunique().reset_index()

providers.rename(columns={'ProviderID': 'Providers'}, inplace=True)

# ==============================
# 🔵 3. MERGE theo ngày
# ==============================

daily = pd.merge(
    patients,
    providers,
    on=['VisitDate', 'Department', 'Hour'],
    how='left'   # 🔥 sửa inner -> left để không mất giờ không có provider
)

# fill provider = 0 nếu không có
daily['Providers'] = daily['Providers'].fillna(0)

# ==============================
# 🔵 4. LẤY TRUNG BÌNH THEO GIỜ + KHOA
# ==============================

final = daily.groupby(['Department', 'Hour'])[
    ['Patients', 'Providers']
].mean().reset_index()

# ==============================
# 🔵 5. RATIO (fix chia 0)
# ==============================

final['Patients_per_Provider'] = final.apply(
    lambda x: x['Patients'] / x['Providers'] if x['Providers'] > 0 else None,
    axis=1
)

# ==============================
# 🔵 6. EXPORT
# ==============================

print(final)
import os

print(os.getcwd())  # xem đang ở đâu

final.to_csv("patients_vs_providers_dept_hour.csv", index=False)


     Department  Hour  Patients  Providers  Patients_per_Provider
0    Cardiology     7  1.071429   1.071429               1.000000
1    Cardiology     8  1.184211   1.184211               1.000000
2    Cardiology     9  1.205882   1.205882               1.000000
3    Cardiology    10  1.279070   1.279070               1.000000
4    Cardiology    11  1.224490   1.224490               1.000000
..          ...   ...       ...        ...                    ...
100   Radiology    12  1.361111   1.361111               1.000000
101   Radiology    13  1.355556   1.333333               1.016667
102   Radiology    14  1.500000   1.500000               1.000000
103   Radiology    15  1.250000   1.250000               1.000000
104   Radiology    16  1.187500   1.187500               1.000000

[105 rows x 5 columns]
g:\My Drive\DUE\Capstone


In [9]:
import pandas as pd

# đọc file
df = pd.read_csv("clean_hospital_data.csv")

# datetime
df['ActualArrivalTime'] = pd.to_datetime(df['ActualArrivalTime'])
df['DischargeTime'] = pd.to_datetime(df['DischargeTime'])

df['VisitDate'] = df['ActualArrivalTime'].dt.date
df['Hour'] = df['ActualArrivalTime'].dt.hour

# ==============================
# 🔵 1. PATIENTS THEO KHOA + GIỜ
# ==============================

patients = df.groupby(['VisitDate','Department','Hour'])['PatientID'] \
    .nunique().reset_index(name='Patients')

# ==============================
# 🔵 2. TOTAL PATIENTS THEO GIỜ
# ==============================

total_patients = patients.groupby(['VisitDate','Hour'])['Patients'] \
    .sum().reset_index(name='TotalPatients')

# ==============================
# 🔵 3. PROVIDERS (toàn viện)
# ==============================

providers = df.groupby(['VisitDate','Hour'])['ProvidersOnShift'] \
    .mean().reset_index(name='TotalProviders')

# ==============================
# 🔵 4. MERGE
# ==============================

data = patients.merge(total_patients, on=['VisitDate','Hour'])
data = data.merge(providers, on=['VisitDate','Hour'])

# ==============================
# 🔵 5. SHARE + PHÂN BỔ PROVIDER
# ==============================

data['PatientShare'] = data['Patients'] / data['TotalPatients']

data['EstimatedProviders'] = data['PatientShare'] * data['TotalProviders']

# ==============================
# 🔵 6. LOAD
# ==============================

data['Load'] = data['Patients'] / data['EstimatedProviders']

# ==============================
# 🔵 7. AVERAGE THEO GIỜ + KHOA
# ==============================

final = data.groupby(['Department','Hour']).agg({
    'Patients': 'mean',
    'EstimatedProviders': 'mean',
    'Load': 'mean'
}).reset_index()

# ==============================
# 🔵 8. EXPORT
# ==============================

final.to_csv("dept_overload_analysis.csv", index=False)

print(final)

     Department  Hour  Patients  EstimatedProviders      Load
0    Cardiology     7  1.071429            2.622381  0.528329
1    Cardiology     8  1.184211            0.879728  1.523803
2    Cardiology     9  1.205882            1.041779  1.387787
3    Cardiology    10  1.279070            1.042664  1.387459
4    Cardiology    11  1.224490            1.083652  1.418097
..          ...   ...       ...                 ...       ...
100   Radiology    12  1.361111            1.113285  1.424654
101   Radiology    13  1.355556            1.171075  1.305172
102   Radiology    14  1.500000            1.157285  1.398211
103   Radiology    15  1.250000            1.053064  1.463300
104   Radiology    16  1.187500            1.378308  1.024555

[105 rows x 5 columns]


In [11]:
import numpy as np
df["ProviderStartTime"] = pd.to_datetime(df["ProviderStartTime"])
df["ProviderEndTime"] = pd.to_datetime(df["ProviderEndTime"])
df["RoomOccupancyTime"] = (
    df["ProviderEndTime"] - df["ProviderStartTime"]
).dt.total_seconds() / 60
room_stats = df.groupby("RoomNumber").agg(
    total_busy_time=("RoomOccupancyTime", "sum"),
    avg_busy_time=("RoomOccupancyTime", "mean"),
    patient_count=("PatientID", "count"),
    first_time=("ProviderStartTime", "min"),
    last_time=("ProviderEndTime", "max")
).reset_index()
room_stats["available_time"] = (
    room_stats["last_time"] - room_stats["first_time"]
).dt.total_seconds() / 60
room_stats["utilization"] = (
    room_stats["total_busy_time"] / room_stats["available_time"]
)

room_stats["utilization"] = room_stats["utilization"].replace([np.inf, -np.inf], np.nan)
bottleneck_rooms = room_stats[
    room_stats["utilization"] >= 0.8
].sort_values("utilization", ascending=False)

print(bottleneck_rooms)


Empty DataFrame
Columns: [RoomNumber, total_busy_time, avg_busy_time, patient_count, first_time, last_time, available_time, utilization]
Index: []


In [6]:
import pandas as pd

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("clean_hospital_data.csv")

# convert datetime
df['ProviderStartTime'] = pd.to_datetime(df['ProviderStartTime'])
df['ProviderEndTime'] = pd.to_datetime(df['ProviderEndTime'])

# =========================
# 2. TẠO DANH SÁCH GIỜ
# =========================

# lấy range ngày
min_time = df['ProviderStartTime'].min().floor('D')
max_time = df['ProviderEndTime'].max().ceil('D')

# tạo tất cả khung giờ
hours = pd.date_range(start=min_time, end=max_time, freq='H')

hour_df = pd.DataFrame({'HourStart': hours})
hour_df['HourEnd'] = hour_df['HourStart'] + pd.Timedelta(hours=1)

# =========================
# 3. CROSS JOIN
# =========================

df['key'] = 1
hour_df['key'] = 1

cross = df.merge(hour_df, on='key').drop('key', axis=1)

# =========================
# 4. FILTER OVERLAP
# =========================

active = cross[
    (cross['ProviderStartTime'] < cross['HourEnd']) &
    (cross['ProviderEndTime'] > cross['HourStart'])
]

# =========================
# 5. ĐẾM PROVIDER
# =========================

active['Date'] = active['HourStart'].dt.date
active['Hour'] = active['HourStart'].dt.hour

active_day_hour = active.groupby(['Date', 'Hour'])['ProviderID'] \
                        .nunique() \
                        .reset_index(name='ActiveProviders')

# =========================
# 6. MEAN THEO GIỜ
# =========================

active_hour = active_day_hour.groupby('Hour')['ActiveProviders'] \
                             .mean() \
                             .reset_index()

print(active_hour)

C:\Users\PC\AppData\Local\Temp\ipykernel_15068\3861596375.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(start=min_time, end=max_time, freq='H')


    Hour  ActiveProviders
0      7         1.000000
1      8         2.853933
2      9         6.131868
3     10         7.000000
4     11         7.142857
5     12         7.252747
6     13         7.439560
7     14         7.142857
8     15         7.263736
9     16         7.373626
10    17         5.164835
11    18         1.883117
12    19         1.130435
13    20         1.111111
14    21         1.000000
15    22         1.000000


C:\Users\PC\AppData\Local\Temp\ipykernel_15068\3861596375.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active['Date'] = active['HourStart'].dt.date
C:\Users\PC\AppData\Local\Temp\ipykernel_15068\3861596375.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active['Hour'] = active['HourStart'].dt.hour


In [15]:
import pandas as pd

# 1. LOAD
df = pd.read_csv("clean_hospital_data.csv")

df['ActualArrivalTime'] = pd.to_datetime(df['ActualArrivalTime'])
df['ProviderStartTime'] = pd.to_datetime(df['ProviderStartTime'])
df['ProviderEndTime'] = pd.to_datetime(df['ProviderEndTime'])

# 2. TIME KEY
df['Date'] = df['ActualArrivalTime'].dt.date
df['Hour'] = df['ActualArrivalTime'].dt.hour

# 3. ONSHIFT
onshift_day_hour = (
    df.groupby(['Date', 'Hour'])['ProvidersOnShift']
      .mean()
      .reset_index()
)

# 4. TIME GRID
time_grid = onshift_day_hour[['Date','Hour']].drop_duplicates().copy()
time_grid['HourStart'] = pd.to_datetime(time_grid['Date']) + pd.to_timedelta(time_grid['Hour'], unit='h')
time_grid['HourEnd'] = time_grid['HourStart'] + pd.Timedelta(hours=1)

# 5. CROSS JOIN
df['key'] = 1
time_grid['key'] = 1

cross = df.merge(time_grid, on='key').drop('key', axis=1)

# 🔥 FIX QUAN TRỌNG
cross['Date'] = cross['Date_y']
cross['Hour'] = cross['Hour_y']

# 6. FILTER OVERLAP
active = cross[
    (cross['ProviderStartTime'] < cross['HourEnd']) &
    (cross['ProviderEndTime'] > cross['HourStart'])
]

# 7. ACTIVE PROVIDERS
active_day_hour = (
    active.groupby(['Date','Hour'])['ProviderID']
          .nunique()
          .reset_index(name='ActiveProviders')
)

# 8. MERGE
compare = pd.merge(
    onshift_day_hour,
    active_day_hour,
    on=['Date','Hour'],
    how='left'
)

compare['ActiveProviders'] = compare['ActiveProviders'].fillna(0)

# 9. FINAL
final = (
    compare.groupby('Hour')
           .agg({
               'ProvidersOnShift': 'mean',
               'ActiveProviders': 'mean'
           })
           .reset_index()
)

final['Utilization'] = final['ActiveProviders'] / final['ProvidersOnShift']

# filter giờ làm việc
final = final[(final['Hour'] >= 7) & (final['Hour'] <= 17)]

print(final)
final.to_csv("providers_analysis_by_hour.csv", index=False, encoding="utf-8-sig")

    Hour  ProvidersOnShift  ActiveProviders  Utilization
0      7          4.896341         0.219512     0.044832
1      8          5.013285         2.791209     0.556762
2      9          5.125798         6.131868     1.196276
3     10          4.938791         7.000000     1.417351
4     11          4.952984         7.142857     1.442132
5     12          4.916157         7.252747     1.475288
6     13          5.073822         7.439560     1.466264
7     14          4.936733         7.188889     1.456204
8     15          5.158923         7.263736     1.407995
9     16          5.024722         7.388889     1.470507
10    17          5.300000         5.400000     1.018868


In [24]:
import pandas as pd

# ==============================
# 1. Load data
# ==============================
df = pd.read_csv('clean_hospital_data.csv')

# ==============================
# 2. Convert datetime
# ==============================
df['ProviderStartTime'] = pd.to_datetime(df['ProviderStartTime'], errors='coerce')
df['ProviderEndTime'] = pd.to_datetime(df['ProviderEndTime'], errors='coerce')

# ==============================
# 3. Tạo cột ngày
# ==============================
df['date'] = df['ProviderStartTime'].dt.date

# ==============================
# 4. SORT (cực kỳ quan trọng cho gap)
# ==============================
df = df.sort_values(['RoomNumber', 'date', 'ProviderStartTime'])

# ==============================
# 5. TÍNH VISITS PER DAY
# ==============================
daily_visits = df.groupby(['RoomNumber', 'date']).size().reset_index(name='visits_per_day')

avg_visits = daily_visits.groupby('RoomNumber')['visits_per_day'].mean().reset_index()

# ==============================
# 6. TÍNH GAP (trong ngày)
# ==============================
df['prev_end_time'] = df.groupby(['RoomNumber', 'date'])['ProviderEndTime'].shift(1)

df['gap_minutes'] = (df['ProviderStartTime'] - df['prev_end_time']).dt.total_seconds() / 60

# ==============================
# 7. GAP TRUNG BÌNH THEO NGÀY
# ==============================
daily_gap = df.groupby(['RoomNumber', 'date'])['gap_minutes'].mean().reset_index()

avg_gap = daily_gap.groupby('RoomNumber')['gap_minutes'].mean().reset_index()

# ==============================
# 8. MERGE 2 METRIC
# ==============================
result = avg_visits.merge(avg_gap, on='RoomNumber', how='outer')

# ==============================
# 9. GIỮ ĐỦ PHÒNG
# ==============================
all_rooms = df[['RoomNumber']].drop_duplicates()

result = all_rooms.merge(result, on='RoomNumber', how='left')

# ==============================
# 10. EXPORT
# ==============================
result.to_csv('room_performance_final.csv', index=False)

# ==============================
# 11. PRINT
# ==============================
print("Tổng số phòng:", result['RoomNumber'].nunique())
print(result)
result.to_csv("room.csv", index=False, encoding="utf-8-sig")

Tổng số phòng: 60
   RoomNumber  visits_per_day  gap_minutes
0         A-1        1.326531   171.100000
1        A-10        1.518519   112.833333
2        A-11        1.640000   149.663194
3        A-12        1.293103   181.700000
4        A-13        1.596491   155.046667
5        A-14        1.589286   136.540000
6        A-15        1.636364   158.425926
7         A-2        1.436364   160.882353
8         A-3        1.741379   183.096154
9         A-4        1.520000   142.547619
10        A-5        1.535714   201.604167
11        A-6        1.574468   162.447368
12        A-7        1.634921   160.191667
13        A-8        1.421053   175.425000
14        A-9        1.666667   170.817500
15        B-1        1.460000   163.676471
16       B-10        1.619048    97.226190
17       B-11        1.418182   170.235294
18       B-12        1.412698   133.305556
19       B-13        1.446429   188.325000
20       B-14        1.440000   185.281250
21       B-15        1.608696   127.